# E006 — Offline evaluation of ranking policies

**Hypotheses:** H11, H6 ([docs/hypotheses.md](../../docs/hypotheses.md))

**Question:** Would ranking by the model's pCTR serve more clicked ads than the policy that produced the logs, and what do safety gates and exploration cost?

**Method:** Reconstruct candidate sets per publisher x hour on the test days, take each creative's share of the cell as its logging propensity, re-score every (impression, candidate) pair with the shipped bundle, and estimate each policy with IPS, SNIPS and doubly robust estimators plus paired hour-block bootstrap lifts.

In [1]:
from IPython.display import Markdown

from charade.analysis.experiment import setup

ctx = setup()
f"mode: {'smoke (fixture, tiny model)' if ctx.smoke else 'full data'}"

'mode: full data'

In [2]:
from charade.analysis import policy_eval
from charade.analysis.experiment import ensure_bundle

ensure_bundle(ctx)
policy_eval.run(ctx.settings, ctx.data_dir, report=ctx.reports / "policy" / "ope.md")
Markdown((ctx.reports / "policy" / "ope.md").read_text())

# Off-policy evaluation on the test split (generated by `uv run poe ope`)

DR uses an independent LightGBM reward model (trained before the last training day), not the evaluated policy's own pCTR.

Evaluated impressions: 111,368 of 127,406 test rows (87.4%); cells: 5,484; mean candidates per cell: 4.13

| Policy | Estimator | CTR | 95 % CI | ESS | Max weight |
|---|---|---|---|---|---|
| logging (observed) | on-policy | 0.1751 | [0.1677, 0.1819] | 111,368 | 1.0 |
| uniform random | ips | 0.1760 | [0.1694, 0.1819] | 17,149 | 196.9 |
| uniform random | snips | 0.1760 | [0.1694, 0.1819] | 17,149 | 196.9 |
| uniform random | dr | 0.1748 | [0.1679, 0.1810] | 17,149 | 196.9 |
| greedy pCTR (no gates) | ips | 0.1869 | [0.1732, 0.1991] | 4,059 | 445.0 |
| greedy pCTR (no gates) | snips | 0.1968 | [0.1832, 0.2099] | 4,059 | 445.0 |
| greedy pCTR (no gates) | dr | 0.1916 | [0.1793, 0.2033] | 4,059 | 445.0 |
| greedy pCTR with gates (no exploration) | ips | 0.1798 | [0.1674, 0.1908] | 4,018 | 445.0 |
| greedy pCTR with gates (no exploration) | snips | 0.1948 | [0.1813, 0.2075] | 4,018 | 445.0 |
| greedy pCTR with gates (no exploration) | dr | 0.1903 | [0.1780, 0.2019] | 4,018 | 445.0 |
| shipped policy (gates + 5% exploration) | ips | 0.1796 | [0.1676, 0.1902] | 4,395 | 425.7 |
| shipped policy (gates + 5% exploration) | snips | 0.1941 | [0.1809, 0.2063] | 4,395 | 425.7 |
| shipped policy (gates + 5% exploration) | dr | 0.1897 | [0.1780, 0.2007] | 4,395 | 425.7 |

## Lift over the logging policy (paired hour-block bootstrap)

| Policy | SNIPS lift [95 % CI] | DR lift [95 % CI] |
|---|---|---|
| uniform random | +0.0009 [-0.0025, +0.0041] | -0.0003 [-0.0036, +0.0026] |
| greedy pCTR (no gates) | +0.0217 [+0.0108, +0.0333] | +0.0165 [+0.0072, +0.0273] |
| greedy pCTR with gates (no exploration) | +0.0197 [+0.0085, +0.0309] | +0.0152 [+0.0052, +0.0250] |
| shipped policy (gates + 5% exploration) | +0.0190 [+0.0084, +0.0297] | +0.0146 [+0.0050, +0.0238] |

## What the shipped policy gives up against ungated greedy (DR, paired hour-block bootstrap)

| Step | CTR cost [95 % CI] |
|---|---|
| gates (brand safety, frequency cap) | +0.0013 [-0.0012, +0.0037] |
| exploration (5 %) | +0.0006 [+0.0001, +0.0011] |
| total | +0.0019 [-0.0006, +0.0045] |


## Reading
- Uniform random lands near the logging CTR, as it should when logging is frequency-proportional.
- Both the ungated greedy and the shipped policy have DR lifts whose intervals exclude zero; the
  shipped policy pays ~0.16 pp for brand safety, caps and exploration.
- The intervals cover sampling noise under the reconstruction assumptions, not bias from them.
- H6 shows up as the frequency-cap gate and as `log_user_campaign_imps` in the model.

## Conclusion
Under reconstructed candidate sets and frequency-share propensities, with DR using an independent LightGBM reward model, the shipped policy's estimated lift over logging is +1.46 pp CTR [+0.50, +2.38] (ungated greedy +1.65 pp; validation day +1.96 pp), on every impression of a multi-creative cell; ESS ~4 % of rows; a demonstration of the evaluation, not a measured production lift.

**Decision:** Ship greedy + gates + 5 % exploration from a closed-form distribution, and log every candidate's exact propensity so the next evaluation does not need reconstruction.